In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_validation import aligned_returns, market_regression


# 09 Systematic Market Risk Diagnostics
Measure market beta, HAC alpha, rolling exposure and stress-day behavior using the benchmark saved in Module 01.


In [ ]:
cfg = ResearchConfig()


In [ ]:
equity = pd.read_parquet("equity_curve.parquet")
benchmark = pd.read_parquet("benchmark_prices.parquet").iloc[:, 0]
rates = pd.read_parquet("risk_free_rates.parquet").iloc[:, 0]
aligned = aligned_returns(equity, benchmark, rates)
aligned.to_parquet("market_alignment.parquet")
market_alpha = market_regression(aligned, cfg.hac_lags)
pd.to_pickle(market_alpha, "market_alpha.pkl")
display(pd.Series(market_alpha))


In [ ]:
records = []
for year, group in aligned.groupby(aligned.index.year):
    records.append({"sample": str(year), **market_regression(group, cfg.hac_lags)})
for label, group in [
    ("market_up", aligned.loc[aligned.market_return > 0]),
    ("market_down", aligned.loc[aligned.market_return <= 0]),
]:
    records.append({"sample": label, **market_regression(group, cfg.hac_lags)})
market_subsamples = pd.DataFrame(records)
market_subsamples.to_parquet("market_subsamples.parquet")
display(market_subsamples)


In [ ]:
rolling = pd.DataFrame(index=aligned.index)
rolling["beta_63"] = (
    aligned.strategy_excess.rolling(63).cov(aligned.market_excess)
    / aligned.market_excess.rolling(63).var()
)
rolling["correlation_63"] = aligned.strategy_return.rolling(63).corr(aligned.market_return)
rolling.to_parquet("rolling_market_exposure.parquet")
rolling.plot(subplots=True, figsize=(11, 5))
plt.tight_layout()
plt.show()

display(aligned.nsmallest(10, "market_return"))
(1 + aligned[["strategy_return", "market_return"]]).cumprod().plot(
    figsize=(11, 4), title="Compounded aligned returns"
)
plt.show()
